# DeepAlpha FreqAI - Quickstart

This notebook walks through the core DeepAlpha pipeline end-to-end, **without requiring Freqtrade**. Perfect for research, prototyping, or learning the primitives before deploying them in FreqAI.

**What we will cover:**
1. Install check
2. Loading / generating sample OHLCV data
3. Triple Barrier labels - with visualisation
4. Training `DeepAlphaModel` components manually
5. SHAP feature importance plot
6. A simple backtest on the test split

For the FreqAI integration, see the main [README](../README.md).

## 1. Install check

In [ ]:
# Uncomment on first run:
# !pip install deepalpha-freqai matplotlib

import deepalpha_freqai
print(f'deepalpha-freqai version: {deepalpha_freqai.__version__}')

## 2. Sample OHLCV data

We generate a realistic price path via geometric Brownian motion plus a handful of standard indicators and a batch of noise features. The noise features give SHAP something to prune.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 5000
returns = rng.normal(0, 0.01, n)
close = 100 * np.exp(np.cumsum(returns))

df = pd.DataFrame({'close': close})
df['ret_1']    = df['close'].pct_change()
df['ret_5']    = df['close'].pct_change(5)
df['sma_10']   = df['close'].rolling(10).mean() / df['close'] - 1
df['sma_50']   = df['close'].rolling(50).mean() / df['close'] - 1
df['vol_20']   = df['ret_1'].rolling(20).std()
df['rsi_like'] = df['ret_1'].rolling(14).apply(
    lambda x: (x[x > 0].sum() / (abs(x).sum() + 1e-9)) * 100, raw=False
)
for k in range(10):
    df[f'noise_{k}'] = rng.normal(0, 1, n)

df = df.dropna().reset_index(drop=True)
print(df.shape)
df.head()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df['close'])
plt.title('Synthetic close price (GBM)')
plt.xlabel('bar')
plt.ylabel('price')
plt.grid(alpha=0.3)
plt.show()

## 3. Triple Barrier labels

Each bar gets a label based on which barrier is touched first within the holding period:
- **+1** - upper barrier (profit taking)
- **-1** - lower barrier (stop loss)
-  **0** - vertical barrier (time expiry)

Barriers are scaled by rolling volatility, so labels adapt to regime shifts.

In [ ]:
from deepalpha_freqai import apply_triple_barrier

labels = apply_triple_barrier(
    df,
    close_col='close',
    profit_taking=2.0,
    stop_loss=1.0,
    max_holding_period=48,
    volatility_window=20,
)

print('Label distribution:')
print(labels.value_counts().sort_index())

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df['close'], color='black', linewidth=0.6, alpha=0.8)
up    = df.index[labels == 1]
down  = df.index[labels == -1]
flat  = df.index[labels == 0]
ax.scatter(up,   df['close'].loc[up],   color='green', s=6, label='+1 profit', alpha=0.6)
ax.scatter(down, df['close'].loc[down], color='red',   s=6, label='-1 loss',   alpha=0.6)
ax.scatter(flat, df['close'].loc[flat], color='grey',  s=3, label='0 neutral', alpha=0.3)
ax.legend()
ax.set_title('Triple Barrier labels on synthetic price')
ax.grid(alpha=0.3)
plt.show()

## 4. Train primary LightGBM

We train a LightGBM classifier on the full feature set - real indicators plus noise. Then we ask SHAP which features actually matter.

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score

feature_cols = [c for c in df.columns if c != 'close']
X, y = df[feature_cols], labels

split = int(len(X) * 0.8)
X_tr, X_te = X[:split], X[split:]
y_tr, y_te = y[:split], y[split:]

primary = LGBMClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=5,
    num_leaves=32, subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1,
)
primary.fit(X_tr, y_tr)

tr_acc = accuracy_score(y_tr, primary.predict(X_tr))
te_acc = accuracy_score(y_te, primary.predict(X_te))
print(f'Train accuracy: {tr_acc:.3f}')
print(f'Test  accuracy: {te_acc:.3f}')

## 5. SHAP feature importance

`select_features_by_shap` samples the training set, computes SHAP values, and returns the top-k features by mean absolute contribution.

In [ ]:
from deepalpha_freqai import select_features_by_shap
import shap

top_features = select_features_by_shap(primary, X_tr, top_k=6)
print('Top features by SHAP:')
for i, f in enumerate(top_features, 1):
    print(f'  {i}. {f}')

# Visualise
explainer = shap.TreeExplainer(primary)
sample    = X_tr.sample(min(500, len(X_tr)), random_state=42)
shap_vals = explainer.shap_values(sample)
shap.summary_plot(shap_vals, sample, plot_type='bar', show=True)

## 6. Simple backtest with meta-labeling intuition

We simulate a naive strategy: go long when the primary model predicts +1, flat otherwise. Then we layer a confidence filter (primary's own probability - a mini meta-label) and compare.

In [ ]:
# Predictions on test split
pred_proba = primary.predict_proba(X_te)
classes    = list(primary.classes_)

# Probability of class +1 (if present)
try:
    prob_up = pred_proba[:, classes.index(1)]
except ValueError:
    prob_up = np.zeros(len(X_te))

future_ret = df['close'].pct_change().shift(-1).loc[X_te.index].fillna(0).values

# Strategy 1: always trade on predicted +1
pred_up = (primary.predict(X_te) == 1).astype(int)
pnl_naive = (pred_up * future_ret).cumsum()

# Strategy 2: only trade when prob_up > 0.55 (meta-confidence filter)
pred_filtered = (pred_up * (prob_up > 0.55)).astype(int)
pnl_filtered  = (pred_filtered * future_ret).cumsum()

plt.figure(figsize=(12, 4))
plt.plot(pnl_naive,    label='Naive (all +1 signals)',   linewidth=1.0)
plt.plot(pnl_filtered, label='Filtered (prob > 0.55)',    linewidth=1.2)
plt.title('Cumulative return - test split')
plt.xlabel('bar')
plt.ylabel('cumulative return')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f'Naive    trades: {pred_up.sum()}')
print(f'Filtered trades: {pred_filtered.sum()}')
print(f'Naive    final PnL: {pnl_naive[-1]:.4f}')
print(f'Filtered final PnL: {pnl_filtered[-1]:.4f}')

## Next steps

- Plug `DeepAlphaModel` into your Freqtrade config - see [README](../README.md).
- Try real OHLCV data (e.g. via `ccxt` or Freqtrade's `download-data`).
- Tune `profit_taking`, `stop_loss`, `max_holding_period` to your timeframe.
- Raise `n_splits` on `PurgedWalkForwardCV` for more rigorous diagnostics.
- Join the [Discord](https://discord.gg/P4yX686m) and share your results.